In [2]:
%pip install pandas numpy scipy statsmodels scikit-learn ambrosia matplotlib
%pip install ambrosia

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.1.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 54.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.1 MB/s  0:00:00
Using cached pybind11-3.1.0-py3-none-any.whl (319 kB)
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=31bc8d2573404c56409e52c42773a5aa23a4dc6fd0e52c3ffe2db98c25df0ded
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib
  Attempting uninstall: hyperopt
    Found existing installation: hyperopt 0.3.0
    Uninstalling hyperopt-0.3.0:
      Successfully uninstalled hyperopt-0.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [ambrosia]


In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.power import TTestIndPower
from sklearn.model_selection import train_test_split
from ambrosia.designer import design_binary

from IPython.display import display

from statsmodels.stats.proportion import proportions_ztest

In [21]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Загрузка данных
df = pd.read_csv("marketplace_activity.csv")

# Преобразование типов
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["gmv"] = pd.to_numeric(df["gmv"], errors="coerce")

print("Размер датасета:", df.shape)
display(df.head())
df.info()

Размер датасета: (238830, 3)


,user_id,date,gmv
0,100005,2026-08-31,0.0000
1,100011,2026-08-31,0.0000
2,100016,2026-08-31,0.0000
3,100048,2026-08-31,0.0000
4,100063,2026-08-31,0.0000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238830 entries, 0 to 238829
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   user_id  238830 non-null  int64         
 1   date     238830 non-null  datetime64[ns]
 2   gmv      238830 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 5.5 MB


In [22]:
# Проверка качества данных

print("Пропуски:")
display(df.isna().sum().to_frame("Количество пропусков"))

print("Количество полных дубликатов:", df.duplicated().sum())

print(
    "Дубликаты по паре user_id + date:",
    df.duplicated(["user_id", "date"]).sum()
)

print("Количество уникальных пользователей:", df["user_id"].nunique())
print("Минимальная дата:", df["date"].min())
print("Максимальная дата:", df["date"].max())
print("Отрицательный GMV:", (df["gmv"] < 0).sum())

# Проверяем соответствие данным из условия
assert df[["user_id", "date", "gmv"]].notna().all().all(), \
    "Обнаружены пропуски в обязательных столбцах."

assert not df.duplicated(["user_id", "date"]).any(), \
    "Есть повторные записи одного пользователя за день."

assert (df["gmv"] >= 0).all(), \
    "Обнаружен отрицательный GMV."

assert df["date"].between(
    "2026-08-31", "2026-10-11"
).all(), "Обнаружены даты вне заданного периода."

print("Проверки пройдены успешно.")

Пропуски:


,Количество пропусков
user_id,0
date,0
gmv,0


Количество полных дубликатов: 0
Дубликаты по паре user_id + date: 0
Количество уникальных пользователей: 57571
Минимальная дата: 2026-08-31 00:00:00
Максимальная дата: 2026-10-11 00:00:00
Отрицательный GMV: 0
Проверки пройдены успешно.


In [23]:
# Параметры эксперимента
alpha = 0.05
beta = 0.20
power = 1 - beta

start_date = pd.Timestamp("2026-08-31")

power_analysis = TTestIndPower()

gmv_results = []
period_metrics = {}

for weeks in range(1, 7):
    end_date = start_date + pd.Timedelta(days=7 * weeks - 1)

    period_df = df[
        df["date"].between(start_date, end_date)
    ]

    # Все пользователи, посетившие площадку за период
    users = period_df.groupby("user_id").agg(
        gmv=("gmv", "sum")
    )

    n = len(users)
    mean_gmv = users["gmv"].mean()
    std_gmv = users["gmv"].std(ddof=1)

    if n < 4 or not np.isfinite(std_gmv) or std_gmv == 0:
        mde_d = np.nan
        mde_rub = np.nan
        mde_percent = np.nan
    else:
        # Размер одной группы при распределении 50/50
        n_per_group = n / 2

        mde_d = power_analysis.solve_power(
            effect_size=None,
            nobs1=n_per_group,
            alpha=alpha,
            power=power,
            ratio=1.0,
            alternative="two-sided"
        )

        mde_rub = mde_d * std_gmv
        mde_percent = (
            mde_rub / mean_gmv * 100
            if mean_gmv > 0 else np.nan
        )

    # Сохраняем данные для расчёта конверсии
    period_metrics[weeks] = users

    gmv_results.append({
        "Неделя": weeks,
        "Конец периода": end_date.date(),
        "Число посетителей": n,
        "Средний GMV, руб.": mean_gmv,
        "Стандартное отклонение, руб.": std_gmv,
        "MDE GMV, руб.": mde_rub,
        "MDE GMV, %": mde_percent
    })

gmv_mde = pd.DataFrame(gmv_results)

display(gmv_mde.round(3))

,Неделя,Конец периода,Число посетителей,"Средний GMV, руб.","Стандартное отклонение, руб.","MDE GMV, руб.","MDE GMV, %"
0,1,2026-09-06,27304,160.0530,334.1120,11.3300,7.0790
1,2,2026-09-13,39981,221.5360,417.6490,11.7040,5.2830
2,3,2026-09-20,47237,280.5540,496.3600,12.7970,4.5610
3,4,2026-09-27,51824,340.4960,573.1630,14.1080,4.1430
4,5,2026-10-04,55084,402.9870,652.9690,15.5890,3.8680
5,6,2026-10-11,57571,462.7750,724.7130,16.9240,3.6570


In [24]:
import inspect
print(inspect.signature(design_binary))
print(inspect.getdoc(design_binary))

(to_design: 'str', prob_a: 'float', sizes: 'Optional[types.SampleSizeType]' = None, effects: 'Optional[types.EffectType]' = None, first_type_errors: 'types.StatErrorType' = (0.05,), second_type_errors: 'types.StatErrorType' = (0.2,), method: 'str' = 'theory', groups_ratio: 'float' = 1.0, alternative: 'str' = 'two-sided', stabilizing_method: 'str' = 'asin', **kwargs) -> 'pd.DataFrame'
Design of experiment parameters for binary metrics based
on a known conversion value.

Parameters
----------
to_design : str
    Parameter to design.
prob_a : float
    Probability of success for the control group.
sizes : SampleSizeType, optional
    List or single value of group sizes.
    For example: ``100``, ``[100, 200]``.
effects : EffectType, optional
    List of single value of relative effects.
    For example: 1.05, [1.05, 1.2].
first_type_errors : StatErrorType, default: ``(0.05, )``
    I type error bounds
    P (detect difference for equal) < alpha.
second_type_errors : StatErrorType, default

In [25]:

conversion_results = []

for weeks in range(1, 7):
    users = period_metrics[weeks].copy()

    n = len(users)
    p = (users["gmv"] > 0).mean()

    # Для бинарной метрики нужен размер каждой группы
    n_per_group = n // 2

    if n_per_group < 2 or p <= 0 or p >= 1:
        conversion_results.append({
            "Неделя": weeks,
            "Конверсия, %": p * 100,
            "MDE конверсии, п.п.": np.nan,
            "MDE конверсии, %": np.nan
        })
        continue

    result = design_binary(
        to_design="effect",
        prob_a=float(p),
        sizes=n_per_group,
        first_type_errors=alpha,
        second_type_errors=beta,
        groups_ratio=1.0,
        alternative="two-sided"
    )

    print(f"\nНеделя {weeks}, базовая конверсия = {p:.4%}")
    print(result)

    conversion_results.append({
        "Неделя": weeks,
        "Конверсия, %": p * 100,
        "Результат Ambrosia": result
    })

conversion_mde = pd.DataFrame(conversion_results)

display(conversion_mde)


Неделя 1, базовая конверсия = 26.0511%
Errors ($\alpha$, $\beta$) (0.05; 0.2)
Group sizes                           
13652                             5.8%

Неделя 2, базовая конверсия = 32.3579%
Errors ($\alpha$, $\beta$) (0.05; 0.2)
Group sizes                           
19990                             4.1%

Неделя 3, базовая конверсия = 37.3817%
Errors ($\alpha$, $\beta$) (0.05; 0.2)
Group sizes                           
23618                             3.3%

Неделя 4, базовая конверсия = 41.6969%
Errors ($\alpha$, $\beta$) (0.05; 0.2)
Group sizes                           
25912                             2.9%

Неделя 5, базовая конверсия = 45.5141%
Errors ($\alpha$, $\beta$) (0.05; 0.2)
Group sizes                           
27542                             2.6%

Неделя 6, базовая конверсия = 48.7832%
Errors ($\alpha$, $\beta$) (0.05; 0.2)
Group sizes                           
28785                             2.4%


,Неделя,"Конверсия, %",Результат Ambrosia
0,1,26.0511,"Errors ($\alpha$, $\beta$) (0.05; 0.2) Group s..."
1,2,32.3579,"Errors ($\alpha$, $\beta$) (0.05; 0.2) Group s..."
2,3,37.3817,"Errors ($\alpha$, $\beta$) (0.05; 0.2) Group s..."
3,4,41.6969,"Errors ($\alpha$, $\beta$) (0.05; 0.2) Group s..."
4,5,45.5141,"Errors ($\alpha$, $\beta$) (0.05; 0.2) Group s..."
5,6,48.7832,"Errors ($\alpha$, $\beta$) (0.05; 0.2) Group s..."


In [26]:
summary = gmv_mde.merge(
    conversion_mde.drop(
        columns=["Результат Ambrosia"],
        errors="ignore"
    ),
    on="Неделя",
    how="left"
)

display(summary.round(3))


,Неделя,Конец периода,Число посетителей,"Средний GMV, руб.","Стандартное отклонение, руб.","MDE GMV, руб.","MDE GMV, %","Конверсия, %"
0,1,2026-09-06,27304,160.0530,334.1120,11.3300,7.0790,26.0510
1,2,2026-09-13,39981,221.5360,417.6490,11.7040,5.2830,32.3580
2,3,2026-09-20,47237,280.5540,496.3600,12.7970,4.5610,37.3820
3,4,2026-09-27,51824,340.4960,573.1630,14.1080,4.1430,41.6970
4,5,2026-10-04,55084,402.9870,652.9690,15.5890,3.8680,45.5140
5,6,2026-10-11,57571,462.7750,724.7130,16.9240,3.6570,48.7830


In [27]:
all_users = df["user_id"].drop_duplicates()

control_users, test_users = train_test_split(
    all_users,
    test_size=0.5,
    random_state=42
)

groups = pd.DataFrame({
    "user_id": list(control_users) + list(test_users),
    "group": (
        ["control"] * len(control_users)
        + ["test"] * len(test_users)
    )
})

print("Число пользователей в контрольной группе:", len(control_users))
print("Число пользователей в тестовой группе:", len(test_users))

display(groups["group"].value_counts())

assert set(control_users).isdisjoint(set(test_users))
assert set(control_users) | set(test_users) == set(all_users)

Число пользователей в контрольной группе: 28785
Число пользователей в тестовой группе: 28786


,count
group,
test,28786
control,28785


In [28]:

hom_start = pd.Timestamp("2026-09-21")
hom_end = pd.Timestamp("2026-10-11")

recent_df = df[
    df["date"].between(hom_start, hom_end)
]

recent_metrics = recent_df.groupby("user_id").agg(
    gmv=("gmv", "sum"),
    purchased=("gmv", lambda x: (x > 0).any())
)

# Присоединяем метрики к полному списку назначенных пользователей
homogeneity = groups.set_index("user_id").join(recent_metrics)

homogeneity["gmv"] = homogeneity["gmv"].fillna(0)
homogeneity["purchased"] = (
    homogeneity["purchased"].fillna(False).astype(bool)
)

control = homogeneity[homogeneity["group"] == "control"]
test = homogeneity[homogeneity["group"] == "test"]

# 1. Сравнение среднего GMV: двусторонний критерий Уэлча
gmv_stat, gmv_pvalue = stats.ttest_ind(
    test["gmv"],
    control["gmv"],
    equal_var=False,
    alternative="two-sided"
)

# 2. Сравнение конверсий: z-тест двух пропорций
successes = np.array([
    test["purchased"].sum(),
    control["purchased"].sum()
])

sizes = np.array([
    len(test),
    len(control)
])

conv_stat, conv_pvalue = proportions_ztest(
    successes,
    sizes,
    alternative="two-sided"
)

# Сводная таблица
results = pd.DataFrame([
    {
        "Метрика": "Средний GMV, руб./пользователя",
        "Контроль": control["gmv"].mean(),
        "Тест": test["gmv"].mean(),
        "Разница": test["gmv"].mean() - control["gmv"].mean(),
        "p-value": gmv_pvalue
    },
    {
        "Метрика": "Конверсия в покупку, %",
        "Контроль": control["purchased"].mean() * 100,
        "Тест": test["purchased"].mean() * 100,
        "Разница, п.п.": (
            test["purchased"].mean()
            - control["purchased"].mean()
        ) * 100,
        "p-value": conv_pvalue
    }
])

display(results.round(4))

/tmp/ipykernel_4983/1118537768.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  homogeneity["purchased"].fillna(False).astype(bool)


,Метрика,Контроль,Тест,Разница,p-value,"Разница, п.п."
0,"Средний GMV, руб./пользователя",233.7226,231.4383,-2.2844,0.5593,NaN
1,"Конверсия в покупку, %",30.7764,30.6086,NaN,0.6625,-0.1678


In [29]:
print("Проверка гомогенности групп при alpha =", alpha)

if gmv_pvalue < alpha:
    print(
        f"GMV: различие статистически значимо, p-value = {gmv_pvalue:.4f}."
    )
else:
    print(
        f"GMV: статистически значимых различий не обнаружено, "
        f"p-value = {gmv_pvalue:.4f}."
    )

if conv_pvalue < alpha:
    print(
        f"Конверсия: различие статистически значимо, "
        f"p-value = {conv_pvalue:.4f}."
    )
else:
    print(
        f"Конверсия: статистически значимых различий не обнаружено, "
        f"p-value = {conv_pvalue:.4f}."
    )

print("\nМаксимально доступная длительность теста — 5 полных недель.")
print("Окончательный выбор длительности нужно обосновать таблицей MDE.")

Проверка гомогенности групп при alpha = 0.05
GMV: статистически значимых различий не обнаружено, p-value = 0.5593.
Конверсия: статистически значимых различий не обнаружено, p-value = 0.6625.

Максимально доступная длительность теста — 5 полных недель.
Окончательный выбор длительности нужно обосновать таблицей MDE.


In [30]:
summary = gmv_mde.merge(
    conversion_mde.drop(
        columns=["Результат Ambrosia"],
        errors="ignore"
    ),
    on="Неделя",
    how="left"
)

display(summary.round(3))

,Неделя,Конец периода,Число посетителей,"Средний GMV, руб.","Стандартное отклонение, руб.","MDE GMV, руб.","MDE GMV, %","Конверсия, %"
0,1,2026-09-06,27304,160.0530,334.1120,11.3300,7.0790,26.0510
1,2,2026-09-13,39981,221.5360,417.6490,11.7040,5.2830,32.3580
2,3,2026-09-20,47237,280.5540,496.3600,12.7970,4.5610,37.3820
3,4,2026-09-27,51824,340.4960,573.1630,14.1080,4.1430,41.6970
4,5,2026-10-04,55084,402.9870,652.9690,15.5890,3.8680,45.5140
5,6,2026-10-11,57571,462.7750,724.7130,16.9240,3.6570,48.7830


In [31]:
print("Размер контрольной группы:", len(control_users))
print("Размер тестовой группы:", len(test_users))

print(
    "Доля контрольной группы:",
    round(len(control_users) / len(all_users) * 100, 2),
    "%"
)

print(
    "Доля тестовой группы:",
    round(len(test_users) / len(all_users) * 100, 2),
    "%"
)

assert set(control_users).isdisjoint(set(test_users))
assert set(control_users) | set(test_users) == set(all_users)

Размер контрольной группы: 28785
Размер тестовой группы: 28786
Доля контрольной группы: 50.0 %
Доля тестовой группы: 50.0 %


In [32]:
print("ИТОГИ ПРОВЕРКИ ГОМОГЕННОСТИ")
print("-" * 40)

if gmv_pvalue < alpha:
    print(
        f"GMV: обнаружено статистически значимое различие "
        f"(p-value = {gmv_pvalue:.4f})."
    )
else:
    print(
        f"GMV: статистически значимых различий не обнаружено "
        f"(p-value = {gmv_pvalue:.4f})."
    )

if conv_pvalue < alpha:
    print(
        f"Конверсия: обнаружено статистически значимое различие "
        f"(p-value = {conv_pvalue:.4f})."
    )
else:
    print(
        f"Конверсия: статистически значимых различий не обнаружено "
        f"(p-value = {conv_pvalue:.4f})."
    )

ИТОГИ ПРОВЕРКИ ГОМОГЕННОСТИ
----------------------------------------
GMV: статистически значимых различий не обнаружено (p-value = 0.5593).
Конверсия: статистически значимых различий не обнаружено (p-value = 0.6625).
